# Exelon data tests

In [1]:
import requests
import pandas as pd
from datetime import timedelta

def get_generation_data(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch generation data from Elexon BMRS API for a given date range.
    Loops through the range in 7-day chunks and returns a combined DataFrame.
    
    Args:
        start_date: Start date in 'YYYY-MM-DD' format
        end_date:   End date in 'YYYY-MM-DD' format
    
    Returns:
        pd.DataFrame with columns: startTime, settlementPeriod, businessType, psrType, quantity
    """
    BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1/generation/actual/per-type"
    
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    all_records = []

    chunk_start = start
    while chunk_start < end:
        chunk_end = min(chunk_start + timedelta(days=7), end)

        params = {
            "from": chunk_start.strftime("%Y-%m-%d"),
            "to": chunk_end.strftime("%Y-%m-%d"),
            "settlementPeriodFrom": 1,
            "settlementPeriodTo": 50,
            "format": "json"
        }

        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        json_data = response.json()

        for period in json_data["data"]:
            for item in period["data"]:
                all_records.append({
                    "startTime":        period["startTime"],
                    "settlementPeriod": period["settlementPeriod"],
                    "businessType":     item["businessType"],
                    "psrType":          item["psrType"],
                    "quantity":         item["quantity"]
                })

        print(f"Fetched: {params['from']} → {params['to']}")
        chunk_start = chunk_end + timedelta(days=1)

    df = pd.DataFrame(all_records)
    df["startTime"] = pd.to_datetime(df["startTime"])
    df = df.sort_values("startTime").reset_index(drop=True)

    return df


# --- Usage ---
df = get_generation_data("2025-01-01", "2025-03-01")
print(df.shape)
print(df.head())

Fetched: 2025-01-01 → 2025-01-08
Fetched: 2025-01-09 → 2025-01-16
Fetched: 2025-01-17 → 2025-01-24
Fetched: 2025-01-25 → 2025-02-01
Fetched: 2025-02-02 → 2025-02-09
Fetched: 2025-02-10 → 2025-02-17
Fetched: 2025-02-18 → 2025-02-25
Fetched: 2025-02-26 → 2025-03-01
(31636, 5)
                  startTime  settlementPeriod businessType  \
0 2025-01-01 00:00:00+00:00                 1   Production   
1 2025-01-01 00:00:00+00:00                 1   Production   
2 2025-01-01 00:00:00+00:00                 1   Production   
3 2025-01-01 00:00:00+00:00                 1   Production   
4 2025-01-01 00:00:00+00:00                 1   Production   

                psrType  quantity  
0               Biomass     880.0  
1            Fossil Gas    3607.0  
2      Fossil Hard coal       0.0  
3            Fossil Oil       0.0  
4  Hydro Pumped Storage       0.0  


In [2]:
df = get_generation_data("2025-01-01", "2025-03-01")
print(df.shape)
print(df.head())

Fetched: 2025-01-01 → 2025-01-08
Fetched: 2025-01-09 → 2025-01-16
Fetched: 2025-01-17 → 2025-01-24
Fetched: 2025-01-25 → 2025-02-01
Fetched: 2025-02-02 → 2025-02-09
Fetched: 2025-02-10 → 2025-02-17
Fetched: 2025-02-18 → 2025-02-25
Fetched: 2025-02-26 → 2025-03-01
(31636, 5)
                  startTime  settlementPeriod businessType  \
0 2025-01-01 00:00:00+00:00                 1   Production   
1 2025-01-01 00:00:00+00:00                 1   Production   
2 2025-01-01 00:00:00+00:00                 1   Production   
3 2025-01-01 00:00:00+00:00                 1   Production   
4 2025-01-01 00:00:00+00:00                 1   Production   

                psrType  quantity  
0               Biomass     880.0  
1            Fossil Gas    3607.0  
2      Fossil Hard coal       0.0  
3            Fossil Oil       0.0  
4  Hydro Pumped Storage       0.0  


In [3]:
df.head(20)

,startTime,settlementPeriod,businessType,psrType,quantity
0,2025-01-01 00:00:00+00:00,1,Production,Biomass,880.000
1,2025-01-01 00:00:00+00:00,1,Production,Fossil Gas,3607.000
2,2025-01-01 00:00:00+00:00,1,Production,Fossil Hard coal,0.000
3,2025-01-01 00:00:00+00:00,1,Production,Fossil Oil,0.000
4,2025-01-01 00:00:00+00:00,1,Production,Hydro Pumped Storage,0.000
5,2025-01-01 00:00:00+00:00,1,Production,Hydro Run-of-river and poundage,736.000
6,2025-01-01 00:00:00+00:00,1,Production,Nuclear,5065.000
7,2025-01-01 00:00:00+00:00,1,Production,Other,183.000
8,2025-01-01 00:00:00+00:00,1,Solar generation,Solar,0.000
9,2025-01-01 00:00:00+00:00,1,Wind generation,Wind Offshore,11444.531


In [4]:
df['startTime']

0       2025-01-01 00:00:00+00:00
1       2025-01-01 00:00:00+00:00
2       2025-01-01 00:00:00+00:00
3       2025-01-01 00:00:00+00:00
4       2025-01-01 00:00:00+00:00
                   ...           
31631   2025-03-01 23:30:00+00:00
31632   2025-03-01 23:30:00+00:00
31633   2025-03-01 23:30:00+00:00
31634   2025-03-01 23:30:00+00:00
31635   2025-03-01 23:30:00+00:00
Name: startTime, Length: 31636, dtype: datetime64[ns, UTC]

In [5]:
def exelon_preproc(df):
    """
    preprocessing exelon dataframe:
    convert StartTime column from object into datetime, pivot PsrType (fuel type) column
    into their own columns with their individual generation quantities,
    """
    df['startTime'] = pd.to_datetime(df['startTime'])
    df_pivot = df.pivot_table(
        index='startTime',
        columns='psrType',
        values='quantity',
        aggfunc='sum'
    )

    df = df.rename(columns={
        "startTime": "datetime"
    })

    df_pivot['TotalOutput-MW'] = df_pivot.sum(axis=1)

    return df_pivot

In [6]:
exelon_df = exelon_preproc(df)
exelon_df

psrType,Biomass,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW
startTime,,,,,,,,,,,,
2025-01-01 00:00:00+00:00,880.0,3607.0,0.0,0.0,0.0,736.0,5065.0,183.0,0.0,11444.531,9113.028,31028.559
2025-01-01 00:30:00+00:00,1078.0,3854.0,0.0,0.0,0.0,745.0,5063.0,290.0,0.0,11138.565,8969.868,31138.433
2025-01-01 01:00:00+00:00,1104.0,3867.0,0.0,0.0,0.0,746.0,5056.0,333.0,0.0,10788.770,8931.922,30826.692
2025-01-01 01:30:00+00:00,1109.0,3726.0,0.0,0.0,0.0,746.0,5060.0,238.0,0.0,10519.534,8810.976,30209.510
2025-01-01 02:00:00+00:00,1064.0,3682.0,0.0,0.0,0.0,747.0,5056.0,277.0,0.0,10706.056,8456.386,29988.442
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-01 21:30:00+00:00,3040.0,14537.0,0.0,0.0,150.0,644.0,4202.0,384.0,0.0,2348.448,2157.832,27463.280
2025-03-01 22:00:00+00:00,3032.0,12731.0,0.0,0.0,0.0,563.0,4195.0,387.0,0.0,2222.523,2184.719,25315.242
2025-03-01 22:30:00+00:00,3036.0,11363.0,0.0,0.0,0.0,546.0,4202.0,319.0,0.0,2386.018,2308.548,24160.566


In [7]:
exelon_df.columns

Index(['Biomass', 'Fossil Gas', 'Fossil Hard coal', 'Fossil Oil',
       'Hydro Pumped Storage', 'Hydro Run-of-river and poundage', 'Nuclear',
       'Other', 'Solar', 'Wind Offshore', 'Wind Onshore', 'TotalOutput-MW'],
      dtype='object', name='psrType')

# Weather data tests

In [8]:
# imports required
from google.cloud import bigquery
import pandas as pd
import requests



def fetch_weather(start_date, end_date, latitude=51.5, longitude=-0.1):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    selected_params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': start_date,
        'end_date': end_date,
        'timezone': 'UTC',
        'hourly': [
        'temperature_2m',
        'wind_speed_100m',
        'wind_gusts_10m',
        'cloud_cover',
        'shortwave_radiation',
        'direct_radiation',
        'diffuse_radiation',
        'pressure_msl',
        'snowfall',
        'rain',
        'precipitation']
        }
        
    response = requests.get(url, params=selected_params, timeout=30)
    response.raise_for_status()
    
    data = response.json()
    
    if "hourly" not in data:
        raise ValueError(f"Unexpected API response:{data}")
    
    df = pd.DataFrame(data["hourly"])
    
    return df

In [9]:
df = fetch_weather('2025-01-01', '2025-03-01')
df.head()

,time,temperature_2m,wind_speed_100m,wind_gusts_10m,cloud_cover,shortwave_radiation,direct_radiation,diffuse_radiation,pressure_msl,snowfall,rain,precipitation
0,2025-01-01T00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
1,2025-01-01T01:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
2,2025-01-01T02:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,0.0,0.0
3,2025-01-01T03:00,11.1,42.1,61.9,100,0.0,0.0,0.0,1011.4,0.0,0.0,0.0
4,2025-01-01T04:00,10.9,43.6,58.3,100,0.0,0.0,0.0,1010.3,0.0,0.0,0.0


In [10]:
def weather_preproc(df):
    ''' preprocess weather dataframe, resample, rename, check quality'''
    # datetime and set index
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')

    # sample half hourly
    df = df.resample('30min').ffill()

    # reset index and rename to datetime
    df = df.reset_index()
    df = df.rename(columns={'time': 'datetime'})

    # rename columns w/ units
    df = df.rename(columns={
        'temperature_2m': 'temperature_2m_c',
        'wind_speed_100m': 'wind_speed_100m_ms',
        'wind_gusts_10m': 'wind_gusts_10m_ms',
        'cloud_cover': 'cloud_cover_pct',
        'shortwave_radiation': 'shortwave_radiation_wm2',
        'direct_radiation': 'direct_radiation_wm2',
        'diffuse_radiation': 'diffuse_radiation_wm2',
        'pressure_msl': 'pressure_msl_hpa',
        'snowfall': 'snowfall_cm',
        'rain': 'rain_mm',
        'precipitation': 'precipitation_mm'
    })
    return df

In [11]:
weather_df = weather_preproc(df)

In [12]:
weather_df.head()

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,rain_mm,precipitation_mm
0,2025-01-01 00:00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
1,2025-01-01 00:30:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
2,2025-01-01 01:00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
3,2025-01-01 01:30:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
4,2025-01-01 02:00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,0.0,0.0


In [13]:
df.dtypes

time                   datetime64[ns]
temperature_2m                float64
wind_speed_100m               float64
wind_gusts_10m                float64
cloud_cover                     int64
shortwave_radiation           float64
direct_radiation              float64
diffuse_radiation             float64
pressure_msl                  float64
snowfall                      float64
rain                          float64
precipitation                 float64
dtype: object

In [14]:
df['time'] = pd.to_datetime(df['time'])
# df = df.set_index('time')


In [15]:
df.dtypes

time                   datetime64[ns]
temperature_2m                float64
wind_speed_100m               float64
wind_gusts_10m                float64
cloud_cover                     int64
shortwave_radiation           float64
direct_radiation              float64
diffuse_radiation             float64
pressure_msl                  float64
snowfall                      float64
rain                          float64
precipitation                 float64
dtype: object

In [16]:
df

,time,temperature_2m,wind_speed_100m,wind_gusts_10m,cloud_cover,shortwave_radiation,direct_radiation,diffuse_radiation,pressure_msl,snowfall,rain,precipitation
0,2025-01-01 00:00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
1,2025-01-01 01:00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
2,2025-01-01 02:00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,0.0,0.0
3,2025-01-01 03:00:00,11.1,42.1,61.9,100,0.0,0.0,0.0,1011.4,0.0,0.0,0.0
4,2025-01-01 04:00:00,10.9,43.6,58.3,100,0.0,0.0,0.0,1010.3,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1435,2025-03-01 19:00:00,5.7,15.5,14.0,0,0.0,0.0,0.0,1035.8,0.0,0.0,0.0
1436,2025-03-01 20:00:00,4.6,12.7,14.4,12,0.0,0.0,0.0,1036.1,0.0,0.0,0.0
1437,2025-03-01 21:00:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
1438,2025-03-01 22:00:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0


In [17]:
df = df.set_index('time')
df

,temperature_2m,wind_speed_100m,wind_gusts_10m,cloud_cover,shortwave_radiation,direct_radiation,diffuse_radiation,pressure_msl,snowfall,rain,precipitation
time,,,,,,,,,,,
2025-01-01 00:00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
2025-01-01 01:00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
2025-01-01 02:00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,0.0,0.0
2025-01-01 03:00:00,11.1,42.1,61.9,100,0.0,0.0,0.0,1011.4,0.0,0.0,0.0
2025-01-01 04:00:00,10.9,43.6,58.3,100,0.0,0.0,0.0,1010.3,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2025-03-01 19:00:00,5.7,15.5,14.0,0,0.0,0.0,0.0,1035.8,0.0,0.0,0.0
2025-03-01 20:00:00,4.6,12.7,14.4,12,0.0,0.0,0.0,1036.1,0.0,0.0,0.0
2025-03-01 21:00:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0


In [18]:
df = df.resample('30min').ffill()
df

,temperature_2m,wind_speed_100m,wind_gusts_10m,cloud_cover,shortwave_radiation,direct_radiation,diffuse_radiation,pressure_msl,snowfall,rain,precipitation
time,,,,,,,,,,,
2025-01-01 00:00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
2025-01-01 00:30:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
2025-01-01 01:00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
2025-01-01 01:30:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
2025-01-01 02:00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2025-03-01 21:00:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2025-03-01 21:30:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2025-03-01 22:00:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0


In [19]:
df = df.reset_index()
df

,time,temperature_2m,wind_speed_100m,wind_gusts_10m,cloud_cover,shortwave_radiation,direct_radiation,diffuse_radiation,pressure_msl,snowfall,rain,precipitation
0,2025-01-01 00:00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
1,2025-01-01 00:30:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
2,2025-01-01 01:00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
3,2025-01-01 01:30:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
4,2025-01-01 02:00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2874,2025-03-01 21:00:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2875,2025-03-01 21:30:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2876,2025-03-01 22:00:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0
2877,2025-03-01 22:30:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0


In [20]:
df = df.rename(columns={'time': 'datetime'})
df

,datetime,temperature_2m,wind_speed_100m,wind_gusts_10m,cloud_cover,shortwave_radiation,direct_radiation,diffuse_radiation,pressure_msl,snowfall,rain,precipitation
0,2025-01-01 00:00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
1,2025-01-01 00:30:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
2,2025-01-01 01:00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
3,2025-01-01 01:30:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
4,2025-01-01 02:00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2874,2025-03-01 21:00:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2875,2025-03-01 21:30:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2876,2025-03-01 22:00:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0
2877,2025-03-01 22:30:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0


In [21]:
df = df.rename(columns={
    'temperature_2m': 'temperature_2m_c',
    'wind_speed_100m': 'wind_speed_100m_ms',
    'wind_gusts_10m': 'wind_gusts_10m_ms',
    'cloud_cover': 'cloud_cover_pct',
    'shortwave_radiation': 'shortwave_radiation_wm2',
    'direct_radiation': 'direct_radiation_wm2',
    'diffuse_radiation': 'diffuse_radiation_wm2',
    'pressure_msl': 'pressure_msl_hpa',
    'snowfall': 'snowfall_cm',
    'rain': 'rain_mm',
    'precipitation': 'precipitation_mm'
})

In [22]:
df

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,rain_mm,precipitation_mm
0,2025-01-01 00:00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
1,2025-01-01 00:30:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
2,2025-01-01 01:00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
3,2025-01-01 01:30:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
4,2025-01-01 02:00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2874,2025-03-01 21:00:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2875,2025-03-01 21:30:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2876,2025-03-01 22:00:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0
2877,2025-03-01 22:30:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0


# Carbon data tests

In [23]:
def load_carbon_intensity_data(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Load Carbon Intensity API data for any specified date range.
    Handles the API 31-day limit by fetching data in chunks.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    if start >= end:
        raise ValueError("end_date must be after start_date")

    dfs = []
    current = start

    while current < end:
        next_date = min(current + timedelta(days=30), end)

        url = (
            f"https://api.carbonintensity.org.uk/intensity/"
            f"{current.strftime('%Y-%m-%d')}/{next_date.strftime('%Y-%m-%d')}"
        )

        response = requests.get(url, timeout=30)
        response.raise_for_status()

        data = response.json().get("data", [])

        if data:
            dfs.append(pd.json_normalize(data))

        current = next_date

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)

In [24]:
df = load_carbon_intensity_data('2025-01-01', '2025-03-01')
df

,from,to,intensity.forecast,intensity.actual,intensity.index
0,2024-12-31T23:30Z,2025-01-01T00:00Z,53.0,51,low
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low
...,...,...,...,...,...
2829,2025-02-28T21:30Z,2025-02-28T22:00Z,223.0,204,high
2830,2025-02-28T22:00Z,2025-02-28T22:30Z,219.0,194,high
2831,2025-02-28T22:30Z,2025-02-28T23:00Z,197.0,172,moderate
2832,2025-02-28T23:00Z,2025-02-28T23:30Z,191.0,165,moderate


In [25]:
df.dtypes

from                   object
to                     object
intensity.forecast    float64
intensity.actual        int64
intensity.index        object
dtype: object

In [26]:
def preprocess_carbon_intensity_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and preprocess Carbon Intensity API dataframe.
    """

    if df.empty:
        return pd.DataFrame(columns=["datetime", "carbon_intensity_gCO2perkWh"])

    df = df.copy()

    df = df.rename(columns={
        "intensity.actual": "actual",
        "intensity.forecast": "forecast"
    })

    df["datetime"] = pd.to_datetime(df["from"], utc=True)
    df["datetime"] = df["datetime"].dt.tz_localize(None)

    df = df.drop(columns=["forecast"], errors="ignore")

    df = df.rename(columns={"actual": "carbon_intensity_gCO2perkWh"})

    df = df[["datetime", "carbon_intensity_gCO2perkWh"]]

    df = (
        df
        .sort_values("datetime")
        .drop_duplicates(subset="datetime")
        .reset_index(drop=True)
    )

    return df

In [27]:
carbon_df = preprocess_carbon_intensity_data(df)
carbon_df

,datetime,carbon_intensity_gCO2perkWh
0,2024-12-31 23:30:00,51
1,2025-01-01 00:00:00,55
2,2025-01-01 00:30:00,54
3,2025-01-01 01:00:00,53
4,2025-01-01 01:30:00,53
...,...,...
2828,2025-02-28 21:30:00,204
2829,2025-02-28 22:00:00,194
2830,2025-02-28 22:30:00,172
2831,2025-02-28 23:00:00,165


# Merging

In [28]:
exelon_df

psrType,Biomass,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW
startTime,,,,,,,,,,,,
2025-01-01 00:00:00+00:00,880.0,3607.0,0.0,0.0,0.0,736.0,5065.0,183.0,0.0,11444.531,9113.028,31028.559
2025-01-01 00:30:00+00:00,1078.0,3854.0,0.0,0.0,0.0,745.0,5063.0,290.0,0.0,11138.565,8969.868,31138.433
2025-01-01 01:00:00+00:00,1104.0,3867.0,0.0,0.0,0.0,746.0,5056.0,333.0,0.0,10788.770,8931.922,30826.692
2025-01-01 01:30:00+00:00,1109.0,3726.0,0.0,0.0,0.0,746.0,5060.0,238.0,0.0,10519.534,8810.976,30209.510
2025-01-01 02:00:00+00:00,1064.0,3682.0,0.0,0.0,0.0,747.0,5056.0,277.0,0.0,10706.056,8456.386,29988.442
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-01 21:30:00+00:00,3040.0,14537.0,0.0,0.0,150.0,644.0,4202.0,384.0,0.0,2348.448,2157.832,27463.280
2025-03-01 22:00:00+00:00,3032.0,12731.0,0.0,0.0,0.0,563.0,4195.0,387.0,0.0,2222.523,2184.719,25315.242
2025-03-01 22:30:00+00:00,3036.0,11363.0,0.0,0.0,0.0,546.0,4202.0,319.0,0.0,2386.018,2308.548,24160.566


In [29]:
weather_df

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,rain_mm,precipitation_mm
0,2025-01-01 00:00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
1,2025-01-01 00:30:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,0.0,0.0
2,2025-01-01 01:00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
3,2025-01-01 01:30:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,0.0,0.0
4,2025-01-01 02:00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2874,2025-03-01 21:00:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2875,2025-03-01 21:30:00,3.7,9.0,8.3,100,0.0,0.0,0.0,1036.3,0.0,0.0,0.0
2876,2025-03-01 22:00:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0
2877,2025-03-01 22:30:00,3.2,8.9,5.4,95,0.0,0.0,0.0,1036.7,0.0,0.0,0.0


In [41]:
carbon_df

,datetime,carbon_intensity_gCO2perkWh
0,2024-12-31 23:30:00,51
1,2025-01-01 00:00:00,55
2,2025-01-01 00:30:00,54
3,2025-01-01 01:00:00,53
4,2025-01-01 01:30:00,53
...,...,...
2828,2025-02-28 21:30:00,204
2829,2025-02-28 22:00:00,194
2830,2025-02-28 22:30:00,172
2831,2025-02-28 23:00:00,165


In [40]:
exelon_df['startTime']

KeyError: 'startTime'

In [34]:
weather_df['datetime']

0      2025-01-01 00:00:00
1      2025-01-01 00:30:00
2      2025-01-01 01:00:00
3      2025-01-01 01:30:00
4      2025-01-01 02:00:00
               ...        
2874   2025-03-01 21:00:00
2875   2025-03-01 21:30:00
2876   2025-03-01 22:00:00
2877   2025-03-01 22:30:00
2878   2025-03-01 23:00:00
Name: datetime, Length: 2879, dtype: datetime64[ns]

In [35]:
carbon_df['datetime']

0      2024-12-31 23:30:00
1      2025-01-01 00:00:00
2      2025-01-01 00:30:00
3      2025-01-01 01:00:00
4      2025-01-01 01:30:00
               ...        
2828   2025-02-28 21:30:00
2829   2025-02-28 22:00:00
2830   2025-02-28 22:30:00
2831   2025-02-28 23:00:00
2832   2025-02-28 23:30:00
Name: datetime, Length: 2833, dtype: datetime64[ns]

In [32]:
df = (weather_df
      .merge(exelon_df, left_on="datetime", right_on="startTime", how="left")
      .drop(columns="startTime")
      .merge(dataset3, on="datetime", how="left"))

ValueError: You are trying to merge on datetime64[ns] and datetime64[ns, UTC] columns for key 'datetime'. If you wish to proceed you should use pd.concat

In [ ]:
# df = (weather_df
#       .merge(exelon_df, on="datetime", how="left")
#       .merge(carbon_df, on="datetime", how="left")
#      )